
# TCP/IP Encapsulation Project – Student Guide (English)

**Goal:** Craft IPv4+TCP packets. On Linux/macOS, use raw sockets; on Windows, automatically fall back to **Scapy + Npcap**. <br>
**Flow:** CSV (Application Messages) → Notebook (Encapsulation Simulation) → Wireshark (Capture) → Report (Explanation) <br>
**Safety:** Educational use only. Prefer loopback/VM. Admin/root privileges usually required.


## Prerequisites
- **Linux/macOS**: `sudo jupyter lab` → capture `lo` in Wireshark → run `demo_send()`
- **Windows**:
  1) Install Wireshark/Npcap with *WinPcap API-compatible mode* + *Support loopback traffic*.
  2) `pip install scapy pandas`
  3) Run Jupyter **as Administrator**
  4) Capture **Npcap Loopback Adapter** in Wireshark
  5) Run `demo_send(iface='Npcap Loopback Adapter')`



## Step 1 — Load Your CSV (Input)
1. Place your CSV file (e.g., `group05_http_input.csv`) in the same folder as this notebook.
2. The CSV must contain the following columns: `msg_id, app_protocol, src_app, dst_app, message, timestamp`.
3. In the next cell, set `CSV_PATH` to your file name and run the cell.
4. Verify that the preview shows your rows correctly.


In [5]:
#TODO: Load CSV file with messages into pandas DataFrame, 

import pandas as pd

filename = "group23_http_input.csv" # "path_to_your_file.csv"
messages_df = pd.read_csv(filename)

messages_df



,app_protocol,src_port,dst_port,message,timestamp
0,HTTP,54321,80,GET /index.html HTTP/1.1,0.010
1,HTTP,80,54321,HTTP/1.1 200 OK,0.045
2,HTTP,54321,80,GET /image.png HTTP/1.1,0.120
3,HTTP,80,54321,HTTP/1.1 404 Not Found,0.160
4,HTTP,54321,80,GET /api/data HTTP/1.1,0.300
5,HTTP,80,54321,HTTP/1.1 200 OK,0.340
6,HTTP,54321,80,POST /login HTTP/1.1 (user=admin&pass=1234),0.250
7,HTTP,80,54321,HTTP/1.1 302 Found (Redirecting...),0.280
8,HTTP,54321,80,GET /dashboard HTTP/1.1,0.350
9,HTTP,80,54321,HTTP/1.1 200 OK (Welcome to Dashboard),0.410



## Step 2 — Validate the Schema
The notebook will automatically check the CSV header. If a required column is missing, you will see an error message.
- If validation fails, fix your CSV and re-run Step 1.
- If it passes, continue to Step 3.


In [6]:

# TODO: Run the cell to validate the CSV file format

def validate_csv_format(df: pd.DataFrame):
    expected_columns = ["app_protocol", "src_port", "dst_port", "message", "timestamp"]
    for col in expected_columns:
        if col not in df.columns:
            raise ValueError(f"Missing expected column: {col}")

messages_df['message'] = messages_df['message'].fillna('') # מילוי הודעות ריקות במחרוזת ריקה למניעת שגיאות


validate_csv_format(messages_df) # ביצוע האימות
print(" CSV schema is valid and matches the new requirements!")

 CSV schema is valid and matches the new requirements!



## Step 3 — Map to TCP/IP Layers (Encapsulation)
This notebook will map each application message to the TCP/IP stack:
- Application → Transport (e.g., TCP/UDP headers)
- Internet (IP headers)
- Link (frame headers/footers)

Just run the cell(s) in this section to see the derived structures.



In [7]:
# בדיקת סביבת העבודה: Windows וזיהוי Scapy

import platform
from scapy.all import IP, TCP, Raw, send, get_if_list

IS_WINDOWS = (platform.system() == 'Windows')

# בדיקה אם Scapy מותקנת ומתפקדת
try:
    from scapy.all import IP, TCP, send
    HAVE_SCAPY = True
    print(f" Scapy is ready. System: {platform.system()}")
except ImportError:
    HAVE_SCAPY = False
    print(" Scapy is not installed. Please run: pip install scapy")

# הדפסת רשימת הממשקים כדי שתוכל לוודא שה-Loopback מופיע
if IS_WINDOWS:
    print("\nAvailable Interfaces (Look for 'Npcap Loopback Adapter'):")
    print(get_if_list())

 Scapy is ready. System: Windows

Available Interfaces (Look for 'Npcap Loopback Adapter'):
['\\Device\\NPF_{6741404C-27CF-4630-8247-B9F74C54AB6E}', '\\Device\\NPF_{9CCF7806-EC43-485B-8FCA-D962E9C78521}', '\\Device\\NPF_{31558AE4-B2A2-49B8-AD0C-D6DABB109BFC}', '\\Device\\NPF_{AA1CBC83-09BA-4715-9943-5A426758E9DB}', '\\Device\\NPF_{C0CF8BB2-9D1F-45AE-BD26-16EC97D9BCD5}', '\\Device\\NPF_{1028E02D-C767-4CD6-B83F-26EC74FB675A}', '\\Device\\NPF_{79B5B2E6-AF01-49F5-83AF-AE7D978B3445}', '\\Device\\NPF_Loopback', '\\Device\\NPF_{E377A9F7-26F2-4535-A3C9-98D63AB34539}']


In [8]:
# Function to calculate checksum

def checksum(data: bytes) -> int:
    if len(data) % 2:
        data += b'\0'
    res = sum(struct.unpack('!%dH' % (len(data)//2), data))
    while res >> 16:
        res = (res & 0xFFFF) + (res >> 16)
    return ~res & 0xFFFF

# Helper function to display the data 
def hexdump(data: bytes, width: int=16):
    for i in range(0, len(data), width):
        chunk = data[i:i+width]
        hex_bytes = ' '.join(f'{b:02x}' for b in chunk)
        ascii_bytes = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)
        print(f"{i:04x}  {hex_bytes:<{width*3}}  {ascii_bytes}")


In [9]:
import socket

def build_ip_header(src_ip: str, dst_ip: str, payload_len: int, proto: int=socket.IPPROTO_TCP) -> bytes:
    version_ihl = (4 << 4) + 5
    tos = 0
    total_length = 20 + payload_len
    identification = random.randint(0, 65535)
    flags_fragment = 0
    ttl = 64
    header_checksum = 0
    src = socket.inet_aton(src_ip)
    dst = socket.inet_aton(dst_ip)
    ip_header = struct.pack('!BBHHHBBH4s4s',
                             version_ihl, tos, total_length, identification,
                             flags_fragment, ttl, proto, header_checksum,
                             src, dst)
    chksum = checksum(ip_header)
    ip_header = struct.pack('!BBHHHBBH4s4s',
                             version_ihl, tos, total_length, identification,
                             flags_fragment, ttl, proto, chksum,
                             src, dst)
    return ip_header


In [10]:
from typing import Optional

def build_tcp_header(src_ip: str, dst_ip: str, src_port: int, dst_port: int, payload: bytes=b'',
                     seq: Optional[int]=None, ack_seq: int=0, flags: int=0x02, window: int=65535) -> bytes:
    if seq is None:
        seq = random.randint(0, 0xFFFFFFFF)
    doff_reserved = (5 << 4)
    checksum_tcp = 0
    urg_ptr = 0
    tcp_header = struct.pack('!HHLLBBHHH',
                              src_port, dst_port, seq, ack_seq,
                              doff_reserved, flags, window,
                              checksum_tcp, urg_ptr)
    placeholder = 0
    protocol = socket.IPPROTO_TCP
    tcp_length = len(tcp_header) + len(payload)
    pseudo_header = struct.pack('!4s4sBBH',
                                socket.inet_aton(src_ip), socket.inet_aton(dst_ip),
                                placeholder, protocol, tcp_length)
    chksum = checksum(pseudo_header + tcp_header + payload)
    
    
    tcp_header = struct.pack('!HHLLBBHHH',
                          src_port, dst_port, seq, ack_seq,
                          doff_reserved, flags, window,
                          chksum, urg_ptr)
    

    return tcp_header


### Cross‑Platform Transport
- Linux/macOS: raw sockets (we include the IP header)
- Windows: Scapy + Npcap fallback (raw TCP sockets are blocked by the OS)

In [11]:
class RawTcpTransport:
    def __init__(self, src_ip: str, dst_ip: str, src_port: int, dst_port: int, iface: Optional[str]=None):
        self.src_ip = src_ip
        self.dst_ip = dst_ip
        self.src_port = src_port
        self.dst_port = dst_port
        self.iface = iface
        self.windows_fallback = IS_WINDOWS
        if not self.windows_fallback:
            self.sock = socket.socket(socket.AF_INET, socket.SOCK_RAW, socket.IPPROTO_RAW)
        else:
            if not HAVE_SCAPY:
                raise RuntimeError(
                    f"Windows detected but Scapy is not available: {SCAPY_IMPORT_ERR}.\n"
                    "Install with: pip install scapy. Ensure Npcap is installed with loopback support."
                )

    def encapsulate(self, data: bytes, flags: int=0x02) -> bytes:
        tcp = build_tcp_header(self.src_ip, self.dst_ip, self.src_port, self.dst_port, data, flags=flags)
        ip  = build_ip_header(self.src_ip, self.dst_ip, len(tcp) + len(data))
        return ip + tcp + data

    def send(self, data: bytes, flags: int=0x02):
        if not self.windows_fallback:
            pkt = self.encapsulate(data, flags=flags)
            self.sock.sendto(pkt, (self.dst_ip, 0))
        else:
            scapy_pkt = SCAPY_IP(src=self.src_ip, dst=self.dst_ip)/SCAPY_TCP(sport=self.src_port, dport=self.dst_port, flags=flags)/SCAPY_Raw(data)
            chosen_iface = self.iface
            if chosen_iface is None and self.dst_ip in ("127.0.0.1", "::1"):
                chosen_iface = "Npcap Loopback Adapter"
            scapy_send(scapy_pkt, verbose=False, iface=chosen_iface)


In [12]:
# find interface name for Windows
if IS_WINDOWS and HAVE_SCAPY:
    try:
        print('\n'.join(get_if_list()))
    except Exception as e:
        print('Could not list interfaces:', e)


\Device\NPF_{6741404C-27CF-4630-8247-B9F74C54AB6E}
\Device\NPF_{9CCF7806-EC43-485B-8FCA-D962E9C78521}
\Device\NPF_{31558AE4-B2A2-49B8-AD0C-D6DABB109BFC}
\Device\NPF_{AA1CBC83-09BA-4715-9943-5A426758E9DB}
\Device\NPF_{C0CF8BB2-9D1F-45AE-BD26-16EC97D9BCD5}
\Device\NPF_{1028E02D-C767-4CD6-B83F-26EC74FB675A}
\Device\NPF_{79B5B2E6-AF01-49F5-83AF-AE7D978B3445}
\Device\NPF_Loopback
\Device\NPF_{E377A9F7-26F2-4535-A3C9-98D63AB34539}


In [13]:
import random, socket, struct # וידוא ספריות בסיסיות

# Preview packet structure
src_ip = '127.0.0.1'
dst_ip = '127.0.0.1'
src_port = random.randint(1024, 65535)
dst_port = 12345
payload = b'Hello Packet (preview)'

# חישוב האורך עבור כותרת ה-IP: 20 בתים של כותרת ה-TCP + אורך הנתונים
total_payload_for_ip = 20 + len(payload)

# בניית החבילה
pkt_preview = build_ip_header(src_ip, dst_ip, total_payload_for_ip) + \
              build_tcp_header(src_ip, dst_ip, src_port, dst_port, payload) + \
              payload

# הצגת המבנה
hexdump(pkt_preview)

0000  45 00 00 3e be 5a 00 00 40 06 be 5d 7f 00 00 01   E..>.Z..@..]....
0010  7f 00 00 01 d3 e3 30 39 64 1b c1 30 00 00 00 00   ......09d..0....
0020  50 02 ff ff 74 8e 00 00 48 65 6c 6c 6f 20 50 61   P...t...Hello Pa
0030  63 6b 65 74 20 28 70 72 65 76 69 65 77 29         cket (preview)



## Step 4 — Capture in Wireshark
1. Start capture in Wireshark.
2. Run the transmit/simulation cells in this notebook.
3. Observe packets appearing in Wireshark (timing may vary by system).
4. Stop the capture and save the file as `.pcap`.

### Suggested Wireshark Filters
- `ip.addr == 127.0.0.1 && tcp.port == 12345`
- `tcp.flags.syn == 1 && tcp.flags.ack == 0 && tcp.port == 12345`
- `tcp.flags.push == 1 && tcp.flags.ack == 1 && tcp.port == 12345`



## Step 5 — Generate/Synthesize Traffic
The notebook will simulate the transmission of your messages. You do not need to change parameters unless instructed in comments.
- Make sure Wireshark is open and ready to capture on your active interface.
- Consider applying a simple filter (e.g., `tcp port 80` for HTTP) to focus the view.


In [14]:
# Create transport instance
import random
import time

# הגדרות שמות כדי למנוע NameError בקלאס המקורי
from scapy.all import IP as SCAPY_IP, TCP as SCAPY_TCP, Raw as SCAPY_Raw, send as scapy_send

src_ip = '127.0.0.1'
dst_ip = '127.0.0.1'
src_port = random.randint(1024, 65535)
dst_port = 12345
iface = "\\Device\\NPF_Loopback"  # Set to "Npcap Loopback Adapter" on Windows if needed
transport = RawTcpTransport(src_ip, dst_ip, src_port, dst_port, iface=iface)

In [15]:
def demo_send(df, delay_sec: float=0.5, flags: int=0x18):
    # הלולאה עוברת על ה-CSV במקום על num_packets
    total = len(df)
    print(f"Sending {total} messages from CSV")
    
    for index, row in df.iterrows():
        # לקיחת ההודעה האמיתית מה-CSV
        payload = str(row['message']).encode('utf-8')
        
        # שליחה דרך הטרנספורט הקיים
        transport.send(payload, flags=flags)
        
        print(f"[{index + 1}/{total}] Sent: {row['message']}")
        time.sleep(delay_sec)

# הרצה על הנתונים שלך
demo_send(messages_df)

Sending 12 messages from CSV


C:\Users\ofirh\AppData\Roaming\Python\Python313\site-packages\scapy\sendrecv.py:485: SyntaxWarning: 'iface' has no effect on L3 I/O send(). For multicast/link-local see https://scapy.readthedocs.io/en/latest/usage.html#multicast
  warnings.warn(


[1/12] Sent: GET /index.html HTTP/1.1
[2/12] Sent: HTTP/1.1 200 OK
[3/12] Sent: GET /image.png HTTP/1.1
[4/12] Sent: HTTP/1.1 404 Not Found
[5/12] Sent: GET /api/data HTTP/1.1
[6/12] Sent: HTTP/1.1 200 OK
[7/12] Sent: POST /login HTTP/1.1 (user=admin&pass=1234)
[8/12] Sent: HTTP/1.1 302 Found (Redirecting...)
[9/12] Sent: GET /dashboard HTTP/1.1
[10/12] Sent: HTTP/1.1 200 OK (Welcome to Dashboard)
[11/12] Sent: GET /api/data HTTP/1.1
[12/12] Sent: HTTP/1.1 200 OK (JSON Data Output)


### Run (commented for safety)
- Linux/macOS: `demo_send()`
- Windows loopback: `demo_send(iface='Npcap Loopback Adapter')`
- Try flags: `flags=0x18` (PSH+ACK), `0x10` (ACK), `0x01` (FIN), `0x04` (RST)


In [16]:
# demo_send(num_packets=3, delay_sec=1.0, flags=0x02)
# demo_send(num_packets=3, flags=0x18)


### Send Messages from CSV file 

Iterate over the rows and send message by message

In [17]:
#Send messages from CSV file

from scapy.all import IP, TCP, Raw, send

# הגדרת הכינויים שהקלאס מצפה להם
SCAPY_IP = IP
SCAPY_TCP = TCP
SCAPY_Raw = Raw
scapy_send = send


for index, row in messages_df.iterrows():
    # Extract message details from the DataFrame row
    message = row['message']
    message = f"test message {index}" if not message else message
    # Send the message using the RawTcpTransport class
    # (You may need to adjust flags and other parameters as needed)
    
    #TODO: uncomment the line below to send the messages
    transport.send(message.encode(), flags=0x18)  # Example with PSH+ACK flags
    
    time.sleep(0.1)  # Optional delay between messages


## Step 6 — Analyze and Explain
In your **report**:
- Explain how the CSV application messages became packets/frames through encapsulation.
- Use **Wireshark screenshots** to illustrate headers, ports, and payloads.
- Link observations back to your CSV rows (e.g., message IDs).



## Deliverables Checklist
- [ ] CSV input file.
- [ ] Executed notebook (with outputs).
- [ ] Wireshark .pcap capture.
